<a href="https://colab.research.google.com/github/dellacortelab/chronosort/blob/main/chronosort_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 

# **Chronosort Colab Pipeline**
Run the Chronosort PCA workflow directly in Google Colab using demo data or your own MMCIF structures.

**Workflow overview**
- Install dependencies and fetch the Chronosort source code
- Use the included lipase demo dataset or upload your own MMCIF files
- Configure PCA parameters and execute the pipeline
- Download the resulting trajectory, eigenvectors, projections, and plots

**Quick start**
1. Run the installation cell
2. Run the dataset cell (demo data is selected by default)
3. Run the analysis cell with default parameters
4. Download your results

**Tips**
- Each code cell can be run with the ▶ button on the left
- The demo dataset contains ~100 lipase structures for quick testing
- To use your own data, uncheck "use_demo_data" in the dataset cell
- Uploads can be MMCIF files or archives (.zip, .tar, etc.)
- Larger uploads may take a few minutes to transfer

**Colab notes**
- Runtime: GPU is optional for this workflow
- Session storage is ephemeral; download results before ending the session

In [ ]:
%%time
#@title Install dependencies and fetch Chronosort
#@markdown This cell clones or updates the Chronosort repository and installs the required Python packages.
import sys
import subprocess
from pathlib import Path

repo_url = "https://github.com/dellacortelab/chronosort.git"
repo_path = Path("chronosort")

if not repo_path.exists():
    print("Cloning Chronosort repository...")
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_path)], check=True)
else:
    print("Updating Chronosort repository...")
    subprocess.run(["git", "pull"], cwd=repo_path, check=True, capture_output=True)

requirements_path = repo_path / "requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)], check=True)
print("Chronosort repository ready.")

In [ ]:
#@title Select input dataset
#@markdown Choose to use the demo lipase dataset or upload your own CIF files.
from pathlib import Path
import io
import zipfile
import tarfile
import shutil

use_demo_data = True  # @param {type:"boolean"}

upload_root = Path("user_uploads")
cif_dir = upload_root / "cif_inputs"

if use_demo_data:
    # Use the demo lipase dataset from the repository
    demo_path = Path("chronosort") / "source" / "lipase" / "lipase_cifs"
    if not demo_path.exists():
        raise RuntimeError("Demo data not found. Please run the installation cell first.")
    
    cif_files = [p for p in sorted(demo_path.glob("*.cif")) if not p.name.startswith("._")]
    if not cif_files:
        raise RuntimeError("No CIF files found in demo dataset.")
    
    CIF_INPUT_DIR = str(demo_path.resolve())
    print(f"Using demo lipase dataset with {len(cif_files)} CIF files from {CIF_INPUT_DIR}")
    for sample in cif_files[:5]:
        print(" -", sample.name)
    if len(cif_files) > 5:
        print(f"... and {len(cif_files) - 5} more")

else:
    # Upload custom CIF files or archives
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("This cell is intended to run inside Google Colab.") from exc

    if cif_dir.exists():
        shutil.rmtree(cif_dir)
    cif_dir.mkdir(parents=True, exist_ok=True)

    def _unique_target(directory, filename):
        base_path = Path(filename).name
        stem = Path(base_path).stem
        suffix = Path(base_path).suffix
        candidate = directory / base_path
        counter = 1
        while candidate.exists():
            candidate = directory / f"{stem}_{counter}{suffix}"
            counter += 1
        return candidate

    def _should_skip(filename: str) -> bool:
        base = Path(filename).name
        return base.startswith("._") or base.startswith("__MACOSX")

    print("Upload one or more .cif files, or archives (.zip, .tar, .tar.gz, .tgz, .tar.bz2) containing them.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No files were uploaded. Please provide at least one CIF or archive.")

    for name, data in uploaded.items():
        if _should_skip(name):
            print(f"Skipped hidden file: {name}")
            continue
        file_path = upload_root / name
        file_path.write_bytes(data)
        lower_name = name.lower()
        if lower_name.endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as zf:
                for member in zf.namelist():
                    if member.endswith("/") or _should_skip(member):
                        continue
                    if member.lower().endswith(".cif"):
                        target = _unique_target(cif_dir, member)
                        with zf.open(member) as source, open(target, "wb") as dest:
                            dest.write(source.read())
        elif lower_name.endswith((".tar", ".tar.gz", ".tgz", ".tar.bz2")):
            with tarfile.open(fileobj=io.BytesIO(data)) as tf:
                for member in tf.getmembers():
                    if not member.isfile() or _should_skip(member.name):
                        continue
                    if member.name.lower().endswith(".cif"):
                        target = _unique_target(cif_dir, member.name)
                        with tf.extractfile(member) as source, open(target, "wb") as dest:
                            dest.write(source.read())
        elif lower_name.endswith(".cif"):
            target = _unique_target(cif_dir, name)
            shutil.copy2(file_path, target)
        else:
            print(f"Skipped unsupported file: {name}")

    cif_files = [p for p in sorted(cif_dir.glob("*.cif")) if not p.name.startswith("._")]
    if not cif_files:
        raise RuntimeError("No CIF files found after processing uploads.")
    CIF_INPUT_DIR = str(cif_dir.resolve())
    print(f"Added {len(cif_files)} CIF files to {CIF_INPUT_DIR}")
    for sample in cif_files[:5]:
        print(" -", sample.name)
    if len(cif_files) > 5:
        print(f"... and {len(cif_files) - 5} more")

In [ ]:
%%time
#@title Run Chronosort PCA pipeline
#@markdown Configure PCA parameters and run the analysis on your selected dataset.
import sys
import subprocess
from pathlib import Path

if "CIF_INPUT_DIR" not in globals():
    raise RuntimeError("No CIF input directory detected. Please run the dataset selection cell first.")

# ===== PCA PARAMETERS =====
PCA_components = "0"  # @param {type:"string"}
#@markdown **PCA components:** Which principal components to use (e.g., "0" for PC1, or "0 1" for PC1+PC2)

animation_scale = 30.0  # @param {type:"number"}
#@markdown **Animation scale:** How far to interpolate along the principal component(s) for visualization

# ===== ADVANCED OPTIONS (optional) =====
custom_input_directory = ""  # @param {type:"string"}
#@markdown **Custom input directory:** Leave empty to use the dataset from the previous cell

output_trajectory_name = "trajectory.pdb"  # @param {type:"string"}
#@markdown **Output trajectory name:** Ordered multi-model PDB of all input structures

output_eigenvectors_name = "vecs.txt"  # @param {type:"string"}
#@markdown **Output eigenvectors name:** Text file containing the principal component vectors

output_animation_name = "projection.pdb"  # @param {type:"string"}
#@markdown **Output animation name:** Multi-model PDB showing motion along selected PC(s)

output_variance_plot_name = "eigenvalues.png"  # @param {type:"string"}
#@markdown **Output variance plot name:** PNG showing variance explained by each principal component

# Parse input directory
cif_dir_path = Path(custom_input_directory.strip()) if custom_input_directory.strip() else Path(CIF_INPUT_DIR)
if not cif_dir_path.exists():
    raise FileNotFoundError(f"Input directory '{cif_dir_path}' does not exist.")

# Parse PCA components
components = []
for piece in PCA_components.replace(";", ",").split(","):
    piece = piece.strip()
    if piece:
        components.append(int(piece))
if not components:
    components = [0]

# Setup output directory
output_root = Path("chronosort_outputs")
output_root.mkdir(parents=True, exist_ok=True)

trajectory_path = output_root / output_trajectory_name
vecs_path = output_root / output_eigenvectors_name
projection_path = output_root / output_animation_name
eigenvalues_path = output_root / output_variance_plot_name

# Build command
repo_root = Path("chronosort")
cmd = [
    sys.executable,
    "scripts/run_analysis.py",
    "--cif_dir",
    str(cif_dir_path),
    "--trajectory_file",
    str(trajectory_path.resolve()),
    "--vecs_file",
    str(vecs_path.resolve()),
    "--projection_file",
    str(projection_path.resolve()),
    "--eigenvalues_file",
    str(eigenvalues_path.resolve()),
    "--scale",
    str(animation_scale),
    "--components",
]
cmd.extend(str(c) for c in components)

print("Running Chronosort analysis...")
print(f"  Input: {cif_dir_path}")
print(f"  PCA components: {components}")
print(f"  Scale: {animation_scale}")
print()

result = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print("stderr:\n" + result.stderr)
    raise RuntimeError(f"Chronosort pipeline failed with exit code {result.returncode}.")

print("\nGenerated files:")
for path in sorted(output_root.glob("*")):
    print(f"  ✓ {path.name}")

In [ ]:
%%time
#@title Download results archive
#@markdown Package the generated outputs into a zip archive and download them locally.
from pathlib import Path
import shutil
import datetime

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("This cell is intended to run inside Google Colab.") from exc

output_root = Path("chronosort_outputs")
if not output_root.exists():
    raise RuntimeError("No outputs found. Run the analysis before downloading.")

timestamp = datetime.datetime.now(datetime.UTC).strftime("%Y%m%d_%H%M%S")
archive_name = f"chronosort_results_{timestamp}"
archive_path = shutil.make_archive(archive_name, "zip", root_dir=output_root)
print(f"Created archive: {archive_path}")
files.download(archive_path)